In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

# Choose a model close to your "4B" requirement
model_id = "Qwen/Qwen2.5-3B-Instruct"

def estimate_real_resources(prompt_length=500):
    # 1. Measure Baseline Model Memory (Weights)
    print(f"Loading {model_id}...")
    torch.cuda.empty_cache()
    start_mem = torch.cuda.memory_allocated()
    
    model = AutoModelForCausalLM.from_pretrained(
        model_id, 
        torch_dtype=torch.float16, 
        device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    
    end_mem = torch.cuda.memory_allocated()
    model_weight_gb = (end_mem - start_mem) / (1024**3)
    print(f"Model weight memory: {model_weight_gb:.2f} GB")

    # 2. Prepare Input
    dummy_text = "Hello " * prompt_length
    inputs = tokenizer(dummy_text, return_tensors="pt", truncation=True, max_length=prompt_length).to("cuda")
    actual_len = inputs.input_ids.shape[1]

    print(inputs.input_ids.shape)

    # 3. Measure Prefill Latency and Activation Memory
    torch.cuda.reset_peak_memory_stats()
    start_time = time.perf_counter()
    
    with torch.no_grad():
        outputs = model(**inputs, use_cache=True)
    
    prefill_time = (time.perf_counter() - start_time) * 1000 # ms
    peak_mem = torch.cuda.max_memory_allocated()
    
    # 4. Extract KV Cache Size
    # past_key_values is a tuple of (key, value) tensors for each layer
    # For Qwen2.5: [layers, 2 (k/v), batch, heads, seq_len, head_dim]
    kv_cache = outputs.past_key_values
    total_kv_elements = sum(k.nelement() + v.nelement() for k, v in kv_cache)
    print(total_kv_elements)
    kv_mem_mb = (total_kv_elements * 2) / (1024**2) # 2 bytes for FP16

    print(f"\n--- Results for {actual_len} tokens ---")
    print(f"Prefill Latency: {prefill_time:.2f} ms")
    print(f"Total KV Cache Memory: {kv_mem_mb:.2f} MB")
    print(f"KV Memory per token: {(kv_mem_mb * 1024 / actual_len):.2f} KB")
    print(f"Peak Memory during Prefill: {peak_mem / (1024**3):.2f} GB")



/home/ssn899/miniforge3/envs/start2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if torch.cuda.is_available():
    estimate_real_resources(500)
else:
    print("This script requires a GPU to measure VRAM accurately.")

Loading Qwen/Qwen2.5-3B-Instruct...


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.50it/s]


Model weight memory: 1.44 GB
torch.Size([1, 500])
9216000

--- Results for 500 tokens ---
Prefill Latency: 455.34 ms
Total KV Cache Memory: 17.58 MB
KV Memory per token: 36.00 KB
Peak Memory during Prefill: 1.62 GB


In [2]:
estimate_real_resources(1)

Loading Qwen/Qwen2.5-3B-Instruct...


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.48it/s]


Model weight memory: 1.44 GB
torch.Size([1, 1])
18432

--- Results for 1 tokens ---
Prefill Latency: 453.74 ms
Total KV Cache Memory: 0.04 MB
KV Memory per token: 36.00 KB
Peak Memory during Prefill: 1.47 GB


In [2]:
estimate_real_resources(10000)

Loading Qwen/Qwen2.5-3B-Instruct...


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.43it/s]


Model weight memory: 1.44 GB
torch.Size([1, 10000])
184320000

--- Results for 10000 tokens ---
Prefill Latency: 569.85 ms
Total KV Cache Memory: 351.56 MB
KV Memory per token: 36.00 KB
Peak Memory during Prefill: 4.40 GB
